<a href="https://colab.research.google.com/github/ergul13/mr_akgul/blob/main/tekno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
import os
from ultralytics import YOLO
import torch
import cv2

BASE_PATH = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
ORIGINAL_PNG = os.path.join(BASE_PATH, 'PNG_Görüntüler')
MODEL_SAVE_DIR = os.path.join(BASE_PATH, 'Gelistirilmis_Model')
FINAL_OUTPUT_DIR = os.path.join(BASE_PATH, 'Saglik_Bakanligi_Kirpilmis')
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

# data.yaml dosyasını labels klasörünün yeni yerine göre güncelliyoruz
yaml_content = f"""
path: {BASE_PATH}
train: PNG_Görüntüler
val: PNG_Görüntüler
nc: 1
names: ['breast']
"""

with open(os.path.join(BASE_PATH, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

In [6]:
import os
import shutil

base_path = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
lbl_src = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Self_Training_Temp/labels/train'
img_src = os.path.join(base_path, 'PNG_Görüntüler')

# Hedef klasörler
train_path = os.path.join(base_path, 'yolo_egitim_seti')
train_imgs = os.path.join(train_path, 'images')
train_lbls = os.path.join(train_path, 'labels')

os.makedirs(train_imgs, exist_ok=True)
os.makedirs(train_lbls, exist_ok=True)

label_files = [f for f in os.listdir(lbl_src) if f.endswith('.txt')]

print(f"{len(label_files)} dosya kopyalanıyor...")
for lbl in label_files:
    shutil.copy(os.path.join(lbl_src, lbl), os.path.join(train_lbls, lbl))

    img_name = lbl.replace('.txt', '.png')
    if os.path.exists(os.path.join(img_src, img_name)):
        shutil.copy(os.path.join(img_src, img_name), os.path.join(train_imgs, img_name))

print("İşlem tamamlandı.")

1328 dosya kopyalanıyor...
İşlem tamamlandı.


In [8]:
import shutil
import os

target_folder = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Saglik_Bakanligi_Kirpilmis'

if os.path.exists(target_folder):
    # Klasörü içeriğiyle birlikte siler
    shutil.rmtree(target_folder)
    # Tekrar boş bir şekilde oluşturur
    os.makedirs(target_folder)
    print("Klasör temizlendi, uzman model için hazır.")

Klasör temizlendi, uzman model için hazır.


In [ ]:
import os
import shutil
import cv2
import torch
from ultralytics import YOLO

# --- YOLLAR ---
BASE_PATH = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
IMG_SRC = os.path.join(BASE_PATH, 'PNG_Görüntüler')
OUTPUT_DIR = os.path.join(BASE_PATH, 'Saglik_Bakanligi_Kirpilmis')
EXPERT_MODEL_PATH = os.path.join(BASE_PATH, 'Gelistirilmis_Model/meme_uzman_model/weights/best.pt')

# 1. ADIM: KLASÖRÜ TEMİZLE
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Klasör temizlendi: {OUTPUT_DIR}")

# 2. ADIM: UZMAN MODELİ YÜKLE
expert_model = YOLO(EXPERT_MODEL_PATH)

# 3. ADIM: TÜM RESİMLERİ YENİDEN KIRP
all_images = os.listdir(IMG_SRC)
total = len(all_images)
print(f"Uzman model ile kırma işlemi sıfırdan başlıyor. Toplam: {total}")

for i, name in enumerate(all_images):
    save_path = os.path.join(OUTPUT_DIR, name)

    img = cv2.imread(os.path.join(IMG_SRC, name))
    if img is None: continue

    # Hassas tespit için conf=0.15
    results = expert_model.predict(img, conf=0.15, verbose=False)

    if len(results[0].boxes) > 0:
        # En yüksek güvenli kutuyu al
        b = results[0].boxes.xyxy[0].tolist()
        x1, y1, x2, y2 = map(int, b)
        h, w = img.shape[:2]
        # Koordinatları sınırlandır ve kırp
        cropped = img[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
        cv2.imwrite(save_path, cropped)
    else:
        # Model tespit edemezse orijinali koru
        cv2.imwrite(save_path, img)

    if (i + 1) % 500 == 0:
        print(f"İlerleme: {i+1}/{total}")
        torch.cuda.empty_cache()

print("İşlem başarıyla tamamlandı. Tüm veri seti uzman modelle güncellendi.")

Klasör temizlendi: /content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Saglik_Bakanligi_Kirpilmis
Uzman model ile kırma işlemi sıfırdan başlıyor. Toplam: 42960
İlerleme: 500/42960
İlerleme: 1000/42960
İlerleme: 1500/42960
İlerleme: 2000/42960
İlerleme: 2500/42960
İlerleme: 3000/42960
İlerleme: 3500/42960


In [ ]:
import os
import pandas as pd
import re

# --- YOLLAR ---
BASE_PATH = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
KIRPILMIS_DIR = os.path.join(BASE_PATH, 'Saglik_Bakanligi_Kirpilmis')
BILGI_YOLU = os.path.join(BASE_PATH, 'Cikarilan_Veriler/Bilgi')
FINAL_CSV_OUTPUT = os.path.join(BASE_PATH, 'siniflandirma_egitim_hazir.csv')

# Excel Dosyaları
excel_files = [
    os.path.join(BILGI_YOLU, 'Supplementary_TRAIN1.xlsx'),
    os.path.join(BILGI_YOLU, 'Supplementary_TRAIN2.xlsx')
]

print("--- SINIFLANDIRMA VERİ SETİ HAZIRLANIYOR ---")

# 1. Excel Verilerini Topla (Etiketler)
meta_list = []
for f in excel_files:
    if os.path.exists(f):
        df_temp = pd.read_excel(f)
        # Sütun isimlerini normalize et (Büyük harf ve boşluk temizliği)
        df_temp.columns = df_temp.columns.str.strip().str.upper()

        # Gereken sütunları seç (Sende 'BI-RADS 0/1/2/4/5' şeklindeydi)
        # Sütun ismini BI_RADS olarak sadeleştiriyoruz
        target_label_col = 'BI-RADS 0/1/2/4/5'
        if target_label_col.upper() in df_temp.columns:
            df_temp = df_temp[['CASENUMBER', target_label_col.upper()]]
            df_temp.columns = ['CaseNumber', 'BI_RADS']
            meta_list.append(df_temp)

df_all_labels = pd.concat(meta_list).drop_duplicates()
df_all_labels['CaseNumber'] = df_all_labels['CaseNumber'].astype(int).astype(str)

# 2. Klasördeki Kırpılmış Görüntüleri Tara
image_paths = []
files = [f for f in os.listdir(KIRPILMIS_DIR) if f.endswith('.png')]

for f in files:
    # Dosya isminden ID'yi çek (Regex ile sayısal kısmı bulur)
    match = re.search(r'_(\d{7,15})_', f)
    if match:
        case_id = match.group(1)
        image_paths.append({
            'CaseNumber': case_id,
            'Full_Path': os.path.join(KIRPILMIS_DIR, f),
            'File_Name': f
        })

df_paths = pd.DataFrame(image_paths)

# 3. Merge (Eşleştirme)
# Resim yolları ile Excel'deki etiketleri vaka numarası üzerinden birleştiriyoruz
final_df = pd.merge(df_paths, df_all_labels, on='CaseNumber', how='inner')

# 4. Kaydet
final_df = final_df[['Full_Path', 'CaseNumber', 'BI_RADS', 'File_Name']]
final_df.to_csv(FINAL_CSV_OUTPUT, index=False)

# 5. Raporlama
print(f"\nİşlem Başarıyla Tamamlandı!")
print(f"Toplam Klasördeki Resim: {len(files)}")
print(f"Eşleşen ve CSV'ye Yazılan: {len(final_df)}")
print(f"Eşleşmeyen (Etiketi bulunamayan) Resim Sayısı: {len(files) - len(final_df)}")
print(f"Final Dosyası: {FINAL_CSV_OUTPUT}")

# İlk 5 satırı kontrol için yazdır
print("\nÖrnek Veri:")
print(final_df.head())

In [ ]:
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt

# Dosya yolu
FINAL_CSV = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/siniflandirma_egitim_hazir.csv'

if os.path.exists(FINAL_CSV):
    df = pd.read_csv(FINAL_CSV)

    # 1. Genel İstatistikler
    print(f"Toplam Satır (Resim) Sayısı: {len(df)}")
    print(f"Benzersiz Vaka Sayısı: {df['CaseNumber'].nunique()}")

    # 2. BI-RADS Dağılımı
    print("\nBI-RADS Kategori Dağılımı:")
    print(df['BI_RADS'].value_counts().sort_index())

    # 3. Dosya Yolu Doğrulama
    df['Path_Exists'] = df['Full_Path'].apply(os.path.exists)
    missing_files = df[df['Path_Exists'] == False]
    if len(missing_files) == 0:
        print("\nTüm dosya yolları geçerli.")
    else:
        print(f"\nHATA: {len(missing_files)} adet dosya yolunda bulunamadı.")

    # 4. Örnek Resimleri Bastırma
    print("\nRastgele Örnekler Görselleştiriliyor...")
    samples = df.sample(min(5, len(df)))
    plt.figure(figsize=(20, 10))

    for i, (idx, row) in enumerate(samples.iterrows()):
        img = cv2.imread(row['Full_Path'])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(1, 5, i + 1)
            plt.imshow(img)
            plt.title(f"ID: {row['CaseNumber']}\nBI-RADS: {row['BI_RADS']}")
            plt.axis('off')
        else:
            print(f"Resim okunamadı: {row['Full_Path']}")

    plt.tight_layout()
    plt.show()
else:
    print("CSV dosyası bulunamadı. Önceki işlemin tamamlandığından emin olun.")